In [ ]:
import os
import cv2
import argparse
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import binary_fill_holes
from skimage.measure import label, regionprops
from skimage.morphology import (
    binary_dilation,
    binary_erosion,
    binary_opening,
    disk,
    remove_small_objects
)


class PatchExtractor:
    def __init__(
        self,
        patch_size=50,
        overlap_step=25,
        min_coverage=0.3,
        non_nucleus_overlap_step=25
    ):
        self.patch_size = patch_size
        self.overlap_step = overlap_step
        self.min_coverage = min_coverage
        self.non_nucleus_overlap_step = non_nucleus_overlap_step
        self.overlap_step_nucleus = overlap_step
        self.overlap_step_non_nucleus = non_nucleus_overlap_step

    def preprocess_image(self, image):
        """
        Preprocess grayscale HT image and extract binary nucleus/cell-region mask.
        """
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        contrast_adjusted = clahe.apply(image)

        filtered = cv2.bilateralFilter(
            contrast_adjusted,
            d=9,
            sigmaColor=75,
            sigmaSpace=75
        )

        _, binary_threshold = cv2.threshold(
            filtered,
            128,
            255,
            cv2.THRESH_BINARY
        )

        filled_image = binary_fill_holes(binary_threshold > 0)
        dilated = binary_dilation(filled_image, disk(15))
        eroded = binary_erosion(dilated, disk(3))
        opened = binary_opening(eroded, disk(5))

        cleaned_mask = remove_small_objects(
            opened,
            min_size=2000
        ).astype(np.uint8)

        return cleaned_mask

    def extract_patches(self, image_path, all_visualize=False):
        """
        Extract overlapping nucleus patches and overlapping non-nucleus patches
        from one grayscale image.

        Returns:
            overlapping_nucleus_patches:
                [((x1, y1, x2, y2), patch, coverage), ...]

            overlapping_non_nucleus_patches:
                [((x, y), patch, coverage), ...]
        """
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if image is None or image.ndim != 2:
            print(f"Skipping {image_path}, not a valid grayscale image.")
            return [], []

        binary_mask = self.preprocess_image(image)

        labeled_image = label(binary_mask)
        regions = regionprops(labeled_image)

        if not regions:
            print(f"No nucleus/cell region found in {image_path}")
            return [], []

        nucleus_region = max(regions, key=lambda r: r.area)
        y_centroid, x_centroid = map(int, nucleus_region.centroid)

        h, w = image.shape
        half_patch = self.patch_size // 2

        # 1. Initial 3x3 nucleus grid around centroid
        nucleus_patches = []

        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                y_start = max(0, y_centroid + dy * self.patch_size - half_patch)
                x_start = max(0, x_centroid + dx * self.patch_size - half_patch)

                y_end = min(y_start + self.patch_size, h)
                x_end = min(x_start + self.patch_size, w)

                # Force full patch size when close to boundary
                if y_end - y_start < self.patch_size:
                    y_start = max(0, y_end - self.patch_size)
                if x_end - x_start < self.patch_size:
                    x_start = max(0, x_end - self.patch_size)

                patch = image[y_start:y_end, x_start:x_end]
                patch_mask = binary_mask[y_start:y_end, x_start:x_end]

                coverage = np.sum(patch_mask > 0) / (self.patch_size * self.patch_size)

                nucleus_patches.append(
                    ((x_start, y_start, x_end, y_end), patch, coverage)
                )

        # 2. Overlapping nucleus patches from 3x3 nucleus region
        overlapping_nucleus_patches = []

        for (x_start, y_start, x_end, y_end), _, _ in nucleus_patches:
            for oy in range(y_start, y_end - self.patch_size + 1, self.overlap_step_nucleus):
                for ox in range(x_start, x_end - self.patch_size + 1, self.overlap_step_nucleus):

                    sub_x_end = ox + self.patch_size
                    sub_y_end = oy + self.patch_size

                    patch = image[oy:sub_y_end, ox:sub_x_end]
                    patch_mask = binary_mask[oy:sub_y_end, ox:sub_x_end]

                    if patch.shape != (self.patch_size, self.patch_size):
                        continue

                    coverage = np.sum(patch_mask > 0) / (self.patch_size * self.patch_size)

                    if coverage >= self.min_coverage:
                        overlapping_nucleus_patches.append(
                            ((ox, oy, sub_x_end, sub_y_end), patch, coverage)
                        )

        # 3. Overlapping non-nucleus patches
        overlapping_non_nucleus_patches = []

        for y in range(0, h - self.patch_size + 1, self.overlap_step_non_nucleus):
            for x in range(0, w - self.patch_size + 1, self.overlap_step_non_nucleus):

                # Skip if patch overlaps with 3x3 nucleus area
                overlaps_nucleus = any(
                    x_start < x + self.patch_size and
                    x < x_end and
                    y_start < y + self.patch_size and
                    y < y_end
                    for (x_start, y_start, x_end, y_end), _, _ in nucleus_patches
                )

                if overlaps_nucleus:
                    continue

                patch = image[y:y + self.patch_size, x:x + self.patch_size]
                patch_mask = binary_mask[y:y + self.patch_size, x:x + self.patch_size]

                if patch.shape != (self.patch_size, self.patch_size):
                    continue

                coverage = np.sum(patch_mask > 0) / (self.patch_size * self.patch_size)

                if coverage >= self.min_coverage:
                    overlapping_non_nucleus_patches.append(
                        ((x, y), patch, coverage)
                    )

        if all_visualize:
            self.visualize_patches(
                image,
                binary_mask,
                nucleus_patches,
                overlapping_nucleus_patches,
                overlapping_non_nucleus_patches
            )

        return overlapping_nucleus_patches, overlapping_non_nucleus_patches






In [ ]:



control_path = "/home/yaganapu/nasdatafolder/senescence/data/control_resized"

senescence_path = "/home/yaganapu/nasdatafolder/senescence/data/h2o2_resized"

output_path = "/home/yaganapu/nasdatafolder/senescence/data/extracted_patches"



extractor = PatchExtractor(
    patch_size=50,
    overlap_step=25,
    min_coverage=0.3,
    non_nucleus_overlap_step=25
)


def process_folder(input_folder, class_name):

    nucleus_dir = os.path.join(output_path, class_name, "overlapping_nucleus")
    non_nucleus_dir = os.path.join(output_path, class_name, "overlapping_non_nucleus")

    os.makedirs(nucleus_dir, exist_ok=True)
    os.makedirs(non_nucleus_dir, exist_ok=True)

    image_files = sorted(os.listdir(input_folder))

    for image_name in image_files:

        image_path = os.path.join(input_folder, image_name)

        base_name = os.path.splitext(image_name)[0]

        # DIRECTLY EXTRACT PATCHES
        overlapping_nucleus_patches, overlapping_non_nucleus_patches = extractor.extract_patches(image_path)

        # SAVE NUCLEUS PATCHES
        for i, (_, patch, _) in enumerate(overlapping_nucleus_patches):

            save_path = os.path.join(
                nucleus_dir,
                f"{base_name}_nucleus_{i}.png"
            )

            cv2.imwrite(save_path, patch)

        # SAVE NON-NUCLEUS PATCHES
        for i, (_, patch, _) in enumerate(overlapping_non_nucleus_patches):

            save_path = os.path.join(
                non_nucleus_dir,
                f"{base_name}_non_nucleus_{i}.png"
            )

            cv2.imwrite(save_path, patch)

        print(f"Finished: {image_name}")


process_folder(control_path, "control")

process_folder(senescence_path, "senescence")

print("Patch extraction completed.")